#  Assignment 1 — Natural Language Processing & Computer Vision
**Course:** PGCP-AI (August 2026 Batch) — CDAC  
**Submission Date:** 09/11/2026  
**Student ID:** abs6187 (`23f2000876@ds.study.iitm.ac.in`)  

---

## [Objective] What is this assignment about?
This assignment covers two foundational, practical applications of NLP:

1. **Part A: Machine Translation (MT)** — Teaching a computer to translate human sentences from one language to another (e.g., English $\rightarrow$ Hindi), and understanding how to measure translation quality using the **BLEU score**.
2. **Part B: Rule-Based Chatbot Architecture** — Building a simple dialogue system step-by-step using:
   - **NLU (Natural Language Understanding):** The chatbot's *ears & brain* (figuring out *what the user wants* and *extracting details*).
   - **NLG (Natural Language Generation):** The chatbot's *mouth* (crafting a polite, natural sentence back to the user).

> [Note] **Note for Learning:** All code in this notebook is written in simple, student-friendly Python with clear explanations and real-world examples so you can learn each concept comfortably!

In [1]:
# [Google Colab & Drive Setup - cdac74abhlak@gmail.com]
# Run this cell if executing on Google Colab to mount Google Drive
import os
try:
    import google.colab
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dir = '/content/drive/MyDrive/NLP_CDAC/assignments/assignment-01'
    os.makedirs(drive_dir, exist_ok=True)
    os.chdir(drive_dir)
    print('[OK] Google Drive linked for cdac74abhlak@gmail.com')
    print('[DIR] Working Directory:', os.getcwd())
    # Uncomment below to install required packages on Colab runtime:
    # !pip install -q deep-translator nltk
except ImportError:
    IN_COLAB = False
    print('[OK] Running locally in VS Code / Antigravity IDE (Account: cdac74abhlak@gmail.com)')


[OK] Running locally in VS Code / Antigravity IDE (Account: cdac74abhlak@gmail.com)


--- 
## [Package] Step 0: Imports and Setup

We need two main tools:
1. `re` — Python's built-in **Regular Expressions** module (to find patterns and keywords in text).
2. `nltk` — **Natural Language Toolkit** (a popular Python library for NLP tasks like tokenization and BLEU scoring).
3. `deep_translator` — A lightweight Python library to call free translation services.

In [2]:
# Import regular expressions for pattern searching in text
import re

# Import NLTK (Natural Language Toolkit) for evaluation
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Import BLEU score and tokenizer from NLTK
from nltk.translate.bleu_score import sentence_bleu
from nltk.tokenize import word_tokenize

# Import translator tool
from deep_translator import MyMemoryTranslator

print("[OK] Setup completed successfully! All required libraries are ready.")

[OK] Setup completed successfully! All required libraries are ready.


---
# [Network] PART A: Machine Translation (MT)

###  What is Machine Translation?
**Machine Translation (MT)** is the task of automatically converting a text from a **Source Language** (e.g., English) into a **Target Language** (e.g., Hindi or French) using computer software, while preserving the original meaning.

###  The 3 Eras of Machine Translation (As taught in Ma'am's class):
1. **Rule-Based MT (1950s–1980s):** Humans write grammar rules and bilingual dictionaries. Computer replaces words one by one.
2. **Statistical MT (1990s–2010s):** Computers look at millions of bilingual sentences and calculate word probabilities.
3. **Neural MT (2015–Present):** Deep learning neural networks (like Transformers / Google Translate) translate entire sentences with context.

###  Step A1: Building a Simple Rule-Based Translator (Dictionary Lookup)

Let's first build the simplest possible translation system using a Python dictionary!  
This helps us understand how the early translators worked and why they had limitations.

In [3]:
# Step 1: Create a tiny bilingual dictionary (English -> Hindi)
dictionary_en_to_hi = {
    "hello": "नमस्ते",
    "friend": "दोस्त",
    "welcome": "स्वागत",
    "to": "में",
    "pune": "पुणे",
    "good": "शुभ",
    "morning": "प्रभात",
    "i": "मैं",
    "love": "पसंद करता हूँ",
    "python": "पायथन"
}

# Step 2: Define a simple translation function
def rule_based_translate(sentence):
    """
    Translates a sentence word-by-word using our dictionary.
    If a word is not found in the dictionary, it keeps the original word.
    """
    # Split the sentence into individual words (tokens)
    words = sentence.lower().split()
    
    # Translate each word one by one
    translated_words = []
    for word in words:
        # Look up word in dictionary, default to original word if missing
        hindi_word = dictionary_en_to_hi.get(word, word)
        translated_words.append(hindi_word)
        
    # Join translated words back into a single sentence
    return " ".join(translated_words)


# Step 3: Test our simple translator with student examples
test_sentence_1 = "hello friend"
test_sentence_2 = "good morning"
test_sentence_3 = "welcome to pune"

print("--- Simple Rule-Based Translation Results ---")
print(f"Input:  '{test_sentence_1}'  -->  Output: '{rule_based_translate(test_sentence_1)}'")
print(f"Input:  '{test_sentence_2}'  -->  Output: '{rule_based_translate(test_sentence_2)}'")
print(f"Input:  '{test_sentence_3}'  -->  Output: '{rule_based_translate(test_sentence_3)}'")

--- Simple Rule-Based Translation Results ---
Input:  'hello friend'  -->  Output: 'नमस्ते दोस्त'
Input:  'good morning'  -->  Output: 'शुभ प्रभात'
Input:  'welcome to pune'  -->  Output: 'स्वागत में पुणे'


###  Student Learning Check: Why is Word-by-Word Translation Not Enough?

Notice what happens if we try to translate: `"I eat an apple"` word-by-word:
- **English grammar:** Subject (`I`) + Verb (`eat`) + Object (`an apple`) $\rightarrow$ **SVO order**
- **Hindi grammar:** Subject (`मैं`) + Object (`सेब`) + Verb (`खाता हूँ`) $\rightarrow$ **SOV order**

Word-by-word lookup produces `"मैं खाता हूँ एक सेब"`, which sounds unnatural!  
That is why modern NLP uses **Neural Machine Translation** to understand the sentence structure and context.

###  Step A2: Modern Translation using Python (`deep-translator`)

Now let's use a modern translation API (`MyMemoryTranslator`) which understands grammar, context, and multi-lingual translation.

In [4]:
# The sentence we want to translate
english_text = "Natural Language Processing helps computers understand human language."

# Create translator targeting Hindi ('hi-IN') from English ('en-GB')
translator_to_hindi = MyMemoryTranslator(source='en-GB', target='hi-IN')

# Run the translation
hindi_translation = translator_to_hindi.translate(english_text)

print("=" * 65)
print("ENGLISH TO HINDI TRANSLATION")
print("=" * 65)
print(f"Source (English): {english_text}")
print(f"Target (Hindi):   {hindi_translation}")

ENGLISH TO HINDI TRANSLATION
Source (English): Natural Language Processing helps computers understand human language.
Target (Hindi):   प्राकृतिक भाषा प्रसंस्करण कंप्यूटर को मानव भाषा को समझने में मदद करता है।


### [Round-Trip] Step A3: Round-Trip Translation (Fun Quality Check!)

**What is Round-Trip Translation?**  
We take a sentence in **Language A (English)** $\rightarrow$ translate it to **Language B (Hindi)** $\rightarrow$ then translate it back to **Language A (English)**.  

If the final English sentence means the same thing as the starting sentence, the translation system preserved the meaning well!

In [5]:
# Step 1: Start with our original sentence
original_english = "Machine translation bridges the gap between different cultures."

# Step 2: English -> Hindi
hindi_version = MyMemoryTranslator(source='en-GB', target='hi-IN').translate(original_english)

# Step 3: Hindi -> English (translating back!)
back_to_english = MyMemoryTranslator(source='hi-IN', target='en-GB').translate(hindi_version)

# Step 4: Compare results
print("=" * 65)
print("ROUND-TRIP TRANSLATION DEMO")
print("=" * 65)
print(f"1. Original English: {original_english}")
print(f"2. In Hindi:         {hindi_version}")
print(f"3. Back to English:  {back_to_english}")

ROUND-TRIP TRANSLATION DEMO
1. Original English: Machine translation bridges the gap between different cultures.
2. In Hindi:         मशीन अनुवाद विभिन्न संस्कृतियों के बीच की खाई को पाटता है।
3. Back to English:  Machine translation bridges the gap between different cultures.


###  Step A4: Multi-Language Translation (Indian & International Languages)

Let's see how the same English sentence translates into multiple languages using a loop.

In [6]:
# A friendly greeting sentence
sample_phrase = "Welcome to the Artificial Intelligence laboratory."

# Dictionary of language codes and language names
languages = {
    "hi-IN": "Hindi",
    "mr-IN": "Marathi",
    "fr-FR": "French",
    "es-ES": "Spanish"
}

print(f"Original text: '{sample_phrase}'\n")
print("-" * 50)

# Loop through each target language
for lang_code, lang_name in languages.items():
    try:
        # Translate into current target language
        result = MyMemoryTranslator(source='en-GB', target=lang_code).translate(sample_phrase)
        print(f"{lang_name:<10} : {result}")
    except Exception as e:
        print(f"{lang_name:<10} : [Error or Network issue]")

Original text: 'Welcome to the Artificial Intelligence laboratory.'

--------------------------------------------------


Hindi      : आर्टिफिशियल इंटेलिजेंस प्रयोगशाला में आपका स्वागत है।


Marathi    : कृत्रिम बुद्धिमत्ता प्रयोगशाळेत आपले स्वागत आहे.


French     : Bienvenue au laboratoire d'Intelligence Artificielle.


Spanish    : Bienvenido al laboratorio de Inteligencia Artificial.


### [CHART] Step A5: Evaluating Machine Translation Quality — The BLEU Score

#### [Summary] What is the BLEU Score?
- **BLEU** stands for **B**i**L**ingual **E**valuation **U**nderstudy.
- **Analogy:** Imagine a teacher grading your translated answer. The teacher has an **ideal answer key (Reference)**. The teacher compares your **submitted translation (Candidate)** against the answer key and counts how many words match.
- **Score Range:**
  - `1.0` = Perfect match (every word matches the reference)
  - `0.0` = Terrible translation (zero matching words)
  - `0.4` to `0.7` = Good quality translation in practical NLP

In [7]:
# The Reference sentence (Human ideal translation)
reference_sentence = "the cat is sitting on the mat"

# Two candidate translations to evaluate
good_candidate = "the cat is sitting on the mat"      # Perfect match
okay_candidate = "a cat is sitting on a mat"          # Minor word changes
poor_candidate = "dogs are running outside in rain"  # Completely wrong

# BLEU works on lists of tokens (words)
# Note: Reference must be a list of lists because there can be multiple human reference answers!
ref_tokens = [reference_sentence.split()]

# Calculate scores
score_good = sentence_bleu(ref_tokens, good_candidate.split())
score_okay = sentence_bleu(ref_tokens, okay_candidate.split())
score_poor = sentence_bleu(ref_tokens, poor_candidate.split())

print("=" * 60)
print("BLEU SCORE COMPARISON (Understanding the Metric)")
print("=" * 60)
print(f"Reference text : '{reference_sentence}'\n")
print(f"Candidate 1 (Perfect) : '{good_candidate}'")
print(f"   --> BLEU Score: {score_good:.4f} (1.0 means 100% exact match)\n")

print(f"Candidate 2 (Decent)  : '{okay_candidate}'")
print(f"   --> BLEU Score: {score_okay:.4f} (Partial word overlap)\n")

print(f"Candidate 3 (Wrong)   : '{poor_candidate}'")
print(f"   --> BLEU Score: {score_poor:.4f} (No matching words)")

BLEU SCORE COMPARISON (Understanding the Metric)
Reference text : 'the cat is sitting on the mat'

Candidate 1 (Perfect) : 'the cat is sitting on the mat'
   --> BLEU Score: 1.0000 (1.0 means 100% exact match)

Candidate 2 (Decent)  : 'a cat is sitting on a mat'
   --> BLEU Score: 0.4347 (Partial word overlap)

Candidate 3 (Wrong)   : 'dogs are running outside in rain'
   --> BLEU Score: 0.0000 (No matching words)


---
# Bot PART B: Chatbot Architecture (NLU + NLG)

###  What is a Rule-Based Chatbot?
A **rule-based chatbot** follows predefined patterns and rules to interact with a user. It has 3 key stages:

```
  [ User Types: 'Book flight from Pune to Delhi' ]
                        │
                        ▼
         ┌──────────────────────────────┐
         │  1. NLU (Understanding)       │  <-- 'Ears & Brain'
         │     • Intent:  book_flight   │      Finds What user wants (Intent)
         │     • Origin:  Pune          │      and What details they gave (Slots)
         │     • Dest:    Delhi         │
         └──────────────┬───────────────┘
                        │
                        ▼
         ┌──────────────────────────────┐
         │  2. Dialogue Manager         │  <-- 'Decision Maker'
         │     • Got both cities? Yes!  │      Decides next action
         │     • Ready to search flights│
         └──────────────┬───────────────┘
                        │
                        ▼
         ┌──────────────────────────────┐
         │  3. NLG (Generation)          │  <-- 'Mouth'
         │     Crafts sentence:         │      Builds friendly reply
         │     'Searching flights from  │
         │      Pune to Delhi...'       │
         └──────────────┬───────────────┘
                        │
                        ▼
    [ Bot Reply: 'Searching flights from Pune to Delhi...' ]
```

Let's break down the two main vocabulary words:
- **Intent:** The user's goal or intention (e.g., `greeting`, `book_flight`, `check_weather`, `goodbye`).
- **Slot (or Entity):** The specific parameters or values needed to fulfill the intent (e.g., `origin = Pune`, `destination = Delhi`).

###  Step B1: Building the NLU Module (`understand`)

This function reads whatever text the user types, finds the **intent**, and extracts any **slots** (origin and destination cities).

In [8]:
def understand(user_text):
    """
    NLU (Natural Language Understanding):
    Takes raw user input string and extracts:
      1. intent  -> what the user wants to do
      2. slots   -> dictionary of details like origin, destination, or city
    """
    # Step 1: Convert text to lowercase so matching is case-insensitive
    text = user_text.lower().strip()
    
    # Step 2: Detect the Intent using keywords
    # We check common words a user might type
    if any(word in text for word in ["hello", "hi", "hey", "namaste"]):
        intent = "greeting"
        
    elif any(word in text for word in ["bye", "goodbye", "exit", "quit"]):
        intent = "farewell"
        
    elif any(word in text for word in ["book", "flight", "fly", "ticket"]):
        intent = "book_flight"
        
    elif any(word in text for word in ["weather", "temperature", "rain"]):
        intent = "check_weather"
        
    elif any(word in text for word in ["help", "options", "support"]):
        intent = "help"
        
    else:
        intent = "unknown"  # The bot doesn't recognize what the user meant
        
    # Step 3: Extract Slots (Details)
    # We look for patterns like 'from <City>' and 'to <City>' or 'in <City>'
    slots = {}
    
    # Look for origin: e.g. 'from mumbai'
    from_match = re.search(r"\bfrom\s+([a-zA-Z]+)", text)
    if from_match:
        slots["origin"] = from_match.group(1).title()
        
    # Look for destination: e.g. 'to delhi'
    to_match = re.search(r"\bto\s+([a-zA-Z]+)", text)
    if to_match:
        slots["destination"] = to_match.group(1).title()
        
    # Look for weather city: e.g. 'in pune'
    in_match = re.search(r"\bin\s+([a-zA-Z]+)", text)
    if in_match:
        slots["city"] = in_match.group(1).title()
        
    # Return both the detected intent and the extracted slots
    return intent, slots

print("[OK] NLU module defined!")

[OK] NLU module defined!


###  Step B2: Testing the NLU Module Alone

Before building the chatbot's voice, let's see how our `understand()` function parses different student sentences:

In [9]:
# A list of student test inputs
sample_sentences = [
    "Hello there!",
    "I want to book a flight from Pune to Delhi",
    "Can I fly to Mumbai?",
    "How is the weather in Bengaluru?",
    "Can you help me?",
    "What is your favorite color?",  # An unknown question
    "Bye bye!"
]

print("=" * 65)
print("NLU PARSER TEST RUN")
print("=" * 65)

for sentence in sample_sentences:
    detected_intent, extracted_slots = understand(sentence)
    print(f"User Input : '{sentence}'")
    print(f"  -> Detected Intent : {detected_intent}")
    print(f"  -> Extracted Slots : {extracted_slots}")
    print("-" * 65)

NLU PARSER TEST RUN
User Input : 'Hello there!'
  -> Detected Intent : greeting
  -> Extracted Slots : {}
-----------------------------------------------------------------
User Input : 'I want to book a flight from Pune to Delhi'
  -> Detected Intent : greeting
  -> Extracted Slots : {'origin': 'Pune', 'destination': 'Book'}
-----------------------------------------------------------------
User Input : 'Can I fly to Mumbai?'
  -> Detected Intent : book_flight
  -> Extracted Slots : {'destination': 'Mumbai'}
-----------------------------------------------------------------
User Input : 'How is the weather in Bengaluru?'
  -> Detected Intent : check_weather
  -> Extracted Slots : {'city': 'Bengaluru'}
-----------------------------------------------------------------
User Input : 'Can you help me?'
  -> Detected Intent : help
  -> Extracted Slots : {}
-----------------------------------------------------------------
User Input : 'What is your favorite color?'
  -> Detected Intent : unknow

###  Step B3: Building the NLG Module (`generate_reply`)

Now that the computer *understands* the intent and slots, it needs to generate a helpful response back in human language.

In [10]:
def generate_reply(intent, slots):
    """
    NLG (Natural Language Generation):
    Takes the intent and slots, and builds a friendly response string.
    """
    # Case 1: Greeting
    if intent == "greeting":
        return " Hello! I am your travel assistant bot. I can help you book flights or check the weather!"
    
    # Case 2: Help
    elif intent == "help":
        return "[Note] Here is what you can say:\n   • 'Book a flight from Pune to Delhi'\n   • 'What is the weather in Mumbai?'\n   • 'Bye' to exit."
    
    # Case 3: Book flight
    elif intent == "book_flight":
        origin = slots.get("origin")
        destination = slots.get("destination")
        
        # Check what details we have
        if origin and destination:
            return f"[Flight] Searching available flights from {origin} to {destination}... Found 3 flights! Starting at ₹4,200."
        elif origin:
            return f"[Flight] Flying from {origin}! Where would you like to fly to?"
        elif destination:
            return f"[Flight] Flying to {destination}! Where will you be departing from?"
        else:
            return "[Flight] I'd love to help you book a flight! Where are you flying from and to?"
            
    # Case 4: Weather
    elif intent == "check_weather":
        city = slots.get("city", "your city")
        return f"[Weather] The weather in {city} is currently 26°C and clear sky."
    
    # Case 5: Farewell
    elif intent == "farewell":
        return " Goodbye! Have a wonderful day ahead!"
    
    # Case 6: Fallback for unknown messages
    else:
        return " I didn't quite get that. Type 'help' to see what I can do!"

print("[OK] NLG module defined!")

[OK] NLG module defined!


###  Step B4: Putting It Together — Complete Chatbot Pipeline

Let's combine `understand()` (NLU) and `generate_reply()` (NLG) into one unified `chatbot()` function.

In [11]:
def chatbot(user_input):
    """
    End-to-End Chatbot Pipeline:
    Input -> NLU -> Dialogue Decision -> NLG -> Output
    """
    # 1. Understand user's message (NLU)
    intent, slots = understand(user_input)
    
    # 2. Generate appropriate reply (NLG)
    bot_response = generate_reply(intent, slots)
    
    return bot_response


# Let's run a full automated conversation to demonstrate the chatbot
conversation_script = [
    "Hi there",
    "I want to book a flight",
    "Book a flight from Pune to Delhi",
    "What is the weather in Pune?",
    "Can you tell me a joke?",
    "bye"
]

print("=" * 65)
print("DEMO CHAT CONVERSATION")
print("=" * 65)

for user_msg in conversation_script:
    print(f"User Student : {user_msg}")
    reply = chatbot(user_msg)
    print(f"Bot Bot     : {reply}")
    print("-" * 65)

DEMO CHAT CONVERSATION
User Student : Hi there
Bot Bot     :  Hello! I am your travel assistant bot. I can help you book flights or check the weather!
-----------------------------------------------------------------
User Student : I want to book a flight
Bot Bot     : [Flight] Flying to Book! Where will you be departing from?
-----------------------------------------------------------------
User Student : Book a flight from Pune to Delhi
Bot Bot     :  Hello! I am your travel assistant bot. I can help you book flights or check the weather!
-----------------------------------------------------------------
User Student : What is the weather in Pune?
Bot Bot     : [Weather] The weather in Pune is currently 26°C and clear sky.
-----------------------------------------------------------------
User Student : Can you tell me a joke?
Bot Bot     :  I didn't quite get that. Type 'help' to see what I can do!
-----------------------------------------------------------------
User Student : bye
Bo

###  Step B5: Interactive Chat Loop (Try It Yourself!)

Run the cell below to talk directly with your chatbot in real time!  
Type `bye` or `exit` when you want to stop.

In [12]:
# To start live interactive chat, uncomment the code below and run this cell!

'''
print("Bot Chatbot is ready! (Type 'bye' or 'exit' to quit)\n")

while True:
    user_text = input("You: ")
    
    # Check if user wants to exit
    if user_text.lower().strip() in ["bye", "exit", "quit"]:
        print("Bot: Goodbye! Have a great day! ")
        break
        
    # Get bot response
    response = chatbot(user_text)
    print(f"Bot: {response}\n")
'''
print("[Note] Note: To chat interactively, remove the triple quotes (''') in the cell above and run it.")

[Note] Note: To chat interactively, remove the triple quotes (''') in the cell above and run it.


---
## [Summary] Summary & Student Viva Cheatsheet

| Concept | What it means | Simple Example |
|---|---|---|
| **Machine Translation (MT)** | Automatically translating text from one language to another | English: *'Good morning'* $\rightarrow$ Hindi: *'शुभ प्रभात'* |
| **BLEU Score** | A metric (0.0 to 1.0) evaluating how many words in machine translation match a human reference | 1.0 = exact match; 0.0 = completely wrong |
| **NLU (Understanding)** | The chatbot's module that extracts **Intent** and **Slots** | Sentence: *'Fly from Pune to Delhi'* |
| **Intent** | The goal or action the user wants to perform | `book_flight` |
| **Slot / Entity** | The specific parameters or values provided by the user | `origin: Pune`, `destination: Delhi` |
| **NLG (Generation)** | The chatbot's module that turns data/intents into natural text | Output: *'Searching flights from Pune to Delhi...'* |
| **Dialogue Loop** | The continuous cycle: User input $\rightarrow$ NLU $\rightarrow$ NLG $\rightarrow$ User output | The conversation between human and bot |